#### Decodificando la Ley: Clasificación Inteligente y Búsqueda Semántica de Jurisprudencia Argentina

##### Descarga del dataset completo

Instalamos primero las librerías que necesitaremos y luego las importamos

In [ ]:
# %pip install huggingface_hub datasets pandas

In [5]:
# Importamos librerías
from huggingface_hub import snapshot_download
import pandas as pd
import os

# # Variables de configuración
# DATASET_PATH = "datos/full-dataset"
# RANDOM_SEED = 42

DATASET_PATH = "gitrepo\\datos\\dataset_sample.jsonl.gz"


Descargamos el dataset completo (~2.5GB)

In [ ]:
# # Importamos librerías
# from huggingface_hub import snapshot_download, login
# import pandas as pd
# import os

# # Autenticación
# HF_TOKEN = ""  # Reemplaza con tu token de Hugging Face
# login(token=HF_TOKEN)

# # Variables de configuración
# DATASET_PATH = "datos/full-dataset"
# RANDOM_SEED = 42

# # Si el dataset ya existe localmente, no lo descargamos de nuevo
# if os.path.isdir(DATASET_PATH):
#     print("El dataset ya ha sido descargado previamente.")
# else:
#     print("Descargando el dataset...")

#     snapshot_download(
#         repo_id="marianbasti/jurisprudencia-Argentina-SAIJ",
#         repo_type="dataset",
#         local_dir=DATASET_PATH,
#         token=HF_TOKEN
#     )

#     print("Descarga finalizada.")

##### Lectura de los datasets

Dataset completo

In [ ]:
# df = pd.read_json(
#     os.path.join(DATASET_PATH, "dataset.jsonl"),
#     lines=True
# )

Dataset de ejemplo (muestra del 1% del dataset completo)

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd
import os
from pathlib import Path

pd.set_option("display.max_rows", None)      # muestra todas las filas
pd.set_option("display.max_columns", None)   # opcional: todas las columnas
pd.set_option("display.max_colwidth", None)  #

DATASET_PATH = Path("datos") / "dataset_sample.jsonl.gz"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en {DATASET_PATH.resolve()}. "
        "Asegurate de que el archivo exista en la carpeta `datos`."
    )

df_sample = pd.read_json(
    DATASET_PATH,
    lines=True,
    compression="gzip"
)

# df_sample = pd.read_json(
#     "dataset_sample.jsonl.gz",
#     lines=True,
#     compression="gzip"
# )

FileNotFoundError: File gitrepo\datos\dataset_sample.jsonl.gz does not exist

In [ ]:
df_sample.head()

In [ ]:


df_sample.head()

#analisis de valores nuilos por columna 
n_filas = len(df_sample)

df_nulos = pd.DataFrame({
    "columna": df_sample.columns,
    "nulos": df_sample.isnull().sum().values,
    "no_nulos": df_sample.notna().sum().values,
    "pct_nulos": df_sample.isnull().sum().values / n_filas * 100,
    "pct_completos": df_sample.notna().sum().values / n_filas * 100,
}).assign(
    tiene_nulos=lambda d: d["nulos"] > 0
).sort_values("pct_nulos", ascending=False).reset_index(drop=True)

# Solo columnas con al menos un valor nulo
df_nulos[df_nulos["tiene_nulos"]]

#dataset sin columnas vacias 
def es_vacio(serie: pd.Series) -> pd.Series:
    """Marca valores nulos (NaN/None) y strings vacíos como vacíos."""
    vacios = serie.isna()
    if serie.dtype == object:
        vacios = vacios | (serie == "")
    return vacios


UMBRAL_VACIOS = 99.97  # redondeado a 2 decimales (ej. 99.965706% -> 99.97%)

pct_vacios = df_sample.apply(lambda col: es_vacio(col).mean() * 100)
columnas_removidas = pct_vacios[pct_vacios.round(2) >= UMBRAL_VACIOS].index.tolist()

df_columnas_removidas = (
    pct_vacios[columnas_removidas]
    .rename("pct_vacios")
    .reset_index()
    .rename(columns={"index": "columna"})
    .sort_values("pct_vacios", ascending=False)
    .reset_index(drop=True)
)

df_sample_limpio = df_sample.drop(columns=columnas_removidas)

print(f"Umbral aplicado: >= {UMBRAL_VACIOS}% vacíos (redondeado)")
print(f"Columnas removidas ({len(columnas_removidas)}):")
print(columnas_removidas)
print(f"\nShape original: {df_sample.shape}")
print(f"Shape limpio:   {df_sample_limpio.shape}")

df_columnas_removidas


df_sample_limpio.columns.to_list()


n_filas = len(df_sample_limpio)

df_nulos = pd.DataFrame({
    "columna": df_sample_limpio.columns,
    "nulos": df_sample_limpio.isnull().sum().values,
    "no_nulos": df_sample_limpio.notna().sum().values,
    "pct_nulos": df_sample_limpio.isnull().sum().values / n_filas * 100,
    "pct_completos": df_sample_limpio.notna().sum().values / n_filas * 100,
}).assign(
    tiene_nulos=lambda d: d["nulos"] > 0
).sort_values("pct_nulos", ascending=False).reset_index(drop=True)

# Solo columnas con al menos un valor nulo
df_nulos[df_nulos["tiene_nulos"]]


total_filas = len(df_sample_limpio)
pais_con_valor = df_sample_limpio["pais"].notna().sum()
pais_nulos = df_sample_limpio["pais"].isna().sum()

print(f"Total filas:        {total_filas}")
print(f"Con país informado: {pais_con_valor} ({pais_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin país):   {pais_nulos} ({pais_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_sample_limpio['pais'].value_counts().sum()}")

df_sample_limpio["pais"].value_counts(dropna=False)


total_filas = len(df_sample_limpio)
provincia_con_valor = df_sample_limpio["provincia"].notna().sum()
provincia_nulos = df_sample_limpio["provincia"].isna().sum()

print(f"Total filas:            {total_filas}")
print(f"Con provincia informada: {provincia_con_valor} ({provincia_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin provincia):   {provincia_nulos} ({provincia_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_sample_limpio['provincia'].value_counts().sum()}")

df_sample_limpio["provincia"].value_counts(dropna=False)


df_sample_limpio.pais.value_counts()


def es_vacio(serie: pd.Series) -> pd.Series:
    vacios = serie.isna()
    if serie.dtype == object:
        vacios = vacios | (serie == "")
    return vacios

sin_provincia = df_sample_limpio[df_sample_limpio["provincia"].isna()]
n = len(sin_provincia)

columnas_con_datos = []
for col in sin_provincia.columns:
    no_vacios = (~es_vacio(sin_provincia[col])).sum()
    if no_vacios > 0:
        columnas_con_datos.append({
            "columna": col,
            "no_vacios": no_vacios,
            "pct_filas": no_vacios / n * 100,
        })

df_cols_sin_provincia = (
    pd.DataFrame(columnas_con_datos)
    .sort_values("no_vacios", ascending=False)
    .reset_index(drop=True)
)

df_cols_sin_provincia


sin_provincia.head()



sin_provincia = es_vacio(df_sample_limpio["provincia"])
sin_fecha = es_vacio(df_sample_limpio["fecha"])
sin_fecha_alta = es_vacio(df_sample_limpio["fecha-alta"])

dropeables = sin_provincia & sin_fecha_alta
resto = sin_provincia & ~sin_fecha_alta

print(f"Dropeables: {dropeables.sum()}")
print(f"Resto sin provincia: {resto.sum()}")
print(f"Quedarían en total: {len(df_sample_limpio) - dropeables.sum()}")

df_sample_limpio[dropeables].shape
df_sample_limpio[resto][["provincia", "fecha", "fecha-alta", "pais", "materia", "texto"]]


df_filtrado = df_sample_limpio[~dropeables]


df_filtrado.shape


total_filas = len(df_filtrado)
provincia_con_valor = df_filtrado["provincia"].notna().sum()
provincia_nulos = df_filtrado["provincia"].isna().sum()

print(f"Total filas:            {total_filas}")
print(f"Con provincia informada: {provincia_con_valor} ({provincia_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin provincia):   {provincia_nulos} ({provincia_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_filtrado['provincia'].value_counts().sum()}")

df_filtrado["provincia"].value_counts(dropna=False)


provincias_extranjeras = [
    "San José de Costa Rica",
    "Ginebra",
    "Bajo Rin",
    "Madrid",
]

df_filtrado[df_filtrado["provincia"].isin(provincias_extranjeras)][
    ["provincia", "pais", "jurisdiccion", "materia", "caratula"]
]


provincias_extranjeras = [
    "San José de Costa Rica",
    "Ginebra",
    "Bajo Rin",
    "Madrid",
]

df_filtrado_arg = df_filtrado[~df_filtrado["provincia"].isin(provincias_extranjeras)]

total_filas = len(df_filtrado_arg)
provincia_con_valor = df_filtrado_arg["provincia"].notna().sum()
provincia_nulos = df_filtrado_arg["provincia"].isna().sum()

print(f"Filas removidas (provincias extranjeras): {len(df_filtrado) - total_filas}")
print(f"Total filas:            {total_filas}")
print(f"Con provincia informada: {provincia_con_valor} ({provincia_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin provincia):   {provincia_nulos} ({provincia_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_filtrado_arg['provincia'].value_counts().sum()}")

df_filtrado_arg["provincia"].value_counts(dropna=False)


import matplotlib.pyplot as plt

conteo_provincia = (
    df_filtrado_arg["provincia"]
    .fillna("Sin provincia")
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 10))
conteo_provincia.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Cantidad de registros por provincia")
ax.set_xlabel("Cantidad")
ax.set_ylabel("Provincia")
plt.tight_layout()
plt.show()

In [ ]:
#analizamos las palabras mas frecuentes en la columna "jurisprudencia" del dataset filtrado

